We preprocess the datasets.
First we load the datasets, we handle different tabular formats: CSV, excel, json, parquet and avro.
Then for every dataset we create the profiles for later processing.

In [1]:
import pandas as pd
import os
import datamart_profiler
from ydata_profiling import ProfileReport



For every tabular format we need a different code for accessing it in a dataframe, so first we want to check the file extension and then based on it we use the right code.
This function reads different formats of datasets and return a df.
Then we can use the df to extract the profiles.

In [2]:
def read_dataset_into_df(path):

    file_ext = os.path.splitext(path)[1].lower()

    try:
        if file_ext == '.csv':
            df = pd.read_csv(path)
        elif file_ext in ('.xls', '.xlsx'):
            df = pd.read_excel(path)
        elif file_ext == '.json':
            df = pd.read_json(path)
        elif file_ext == '.parquet':
            df = pd.read_parquet(path)
        elif file_ext == '.feather':
            df = pd.read_feather(path)
        else:
            print(f"Skipping unsupported file: {path}")
            return None

        return df

    except Exception as e:
        print(f"Error reading {path}: {e}")
        return None

Now we define a function to generate the data profile from the df

In [3]:
def generate_data_profile(df, file_path):

    metadata = datamart_profiler.process_dataset(df)

    profile_summary = []

    # Iterate through each column in the metadata and summarize the details
    for column_meta in metadata['columns']:
        column_summary = f"**{column_meta['name']}**: "
        
        # Structural type
        structural_type = column_meta.get('structural_type', 'Unknown')
        column_summary += f"Data is of type {structural_type.split('/')[-1].lower()}. "

        # Number of distinct values (if applicable)
        if 'num_distinct_values' in column_meta:
            num_distinct_values = column_meta['num_distinct_values']
            column_summary += f"There are {num_distinct_values} unique values. "

        # Check if the column is numeric
        if pd.api.types.is_numeric_dtype(df[column_meta['name']]):
            column_summary += "This column is numeric. " 
            mean_value = df[column_meta['name']].mean()
            max_value = df[column_meta['name']].max()
            min_value = df[column_meta['name']].min()
            column_summary += f"Mean: {mean_value}, Max: {max_value}, Min: {min_value}. "
        elif pd.api.types.is_datetime64_any_dtype(df[column_meta['name']]):
            column_summary += "This column is of datetime type. "
            # Calculate and add the range of dates
            min_date = df[column_meta['name']].min()
            max_date = df[column_meta['name']].max()
            column_summary += f"Date range: from {min_date} to {max_date}. "
        else:
            # For categorical columns, get the top 3 most frequent categories
            value_counts = df[column_meta['name']].value_counts()
            if value_counts.nunique() > 1:
                top_categories = df[column_meta['name']].value_counts().nlargest(3).index.tolist()
                column_summary += f"Top 3 frequent values: {', '.join(top_categories)}. "

        # Handle coverage (if available)
        if 'coverage' in column_meta:
            low=0
            high=0
            for i in range(len(column_meta['coverage'])):
                if(column_meta['coverage'][i]['range']['gte']<low):
                    low=column_meta['coverage'][i]['range']['gte']
                if(column_meta['coverage'][i]['range']['lte']>high):
                    high=column_meta['coverage'][i]['range']['lte']
            column_summary += f"Coverage spans from {low} to {high}. "

        # Append the summarized profile for this column
        profile_summary.append(column_summary)

    dataset_title = os.path.splitext(os.path.basename(file_path))[0]

    final_profile_summary = "The key data profile information for the dataset "+ dataset_title +" includes:\n" + '\n'.join(profile_summary)

    return final_profile_summary

In [4]:
dataset_dir = os.path.join("..", "test_preprocessing_datasets")

if not os.path.isdir(dataset_dir):
    print(f"Directory not found: {dataset_dir}")
else:
    for filename in os.listdir(dataset_dir):
        dataset = os.path.join(dataset_dir, filename)
        if not os.path.isfile(dataset):
            continue

        df = read_dataset_into_df(dataset)
        if df is not None:
            print(generate_data_profile(df, dataset))

The key data profile information for the dataset drake_data includes:
**album**: Data is of type text. There are 18 unique values. Top 3 frequent values: Unreleased Songs, Scorpion, Comeback Season. 
**lyrics_title**: Data is of type text. Top 3 frequent values: Do What You Do (Remix) (Ft. Nickelus F & No Malice) Lyrics, City is Mine Lyrics, Houstatlantavegas Lyrics. 
**lyrics_url**: Data is of type text. There are 278 unique values. Top 3 frequent values: https://genius.com/Drake-do-what-you-do-remix-lyrics, https://genius.com/Drake-city-is-mine-lyrics, https://genius.com/Drake-houstatlantavegas-lyrics. 
**lyrics**: Data is of type text. Top 3 frequent values: [Intro: Drake]
I do this for ya'll man
Malice, Nickelus F, it's the REMIX!

[Verse 1: Malice]
Lights slow motion, steppin' out the Phantom
All eyes on me, peekin' and glancin'
Leanin' on the wall, with b-boys dancin'
Ears, neck, and wrists lookin' like the king's ransom
One chick, each arm, makin' it a tandem
Flat here costing' 

c:\Users\rite\AppData\Local\anaconda3\Lib\site-packages\datamart_profiler\core.py:199: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data = data.astype(object).fillna('').astype(str)
c:\Users\rite\AppData\Local\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\rite\AppData\Local\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get

The key data profile information for the dataset electric_vehicles_spec_2025.csv includes:
**brand**: Data is of type text. There are 59 unique values. Top 3 frequent values: Mercedes-Benz, Audi, Porsche. 
**model**: Data is of type text. There are 477 unique values. 
**top_speed_kmh**: Data is of type integer. There are 38 unique values. This column is numeric. Mean: 185.48744769874477, Max: 325, Min: 125. Coverage spans from 0 to 290.0. 
**battery_capacity_kWh**: Data is of type float. This column is numeric. Mean: 74.04372384937238, Max: 118.0, Min: 21.3. Coverage spans from 0 to 118.0. 
**battery_type**: Data is of type text. There are 1 unique values. 
**number_of_cells**: Data is of type integer. There are 38 unique values. This column is numeric. Mean: 485.29347826086956, Max: 7920.0, Min: 72.0. Coverage spans from 0 to 396.0. 
**torque_nm**: Data is of type integer. There are 128 unique values. This column is numeric. Mean: 498.0127388535032, Max: 1350.0, Min: 113.0. Coverage s

c:\Users\rite\AppData\Local\anaconda3\Lib\site-packages\datamart_profiler\core.py:199: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data = data.astype(object).fillna('').astype(str)


The key data profile information for the dataset flights_RUH includes:
**flight_number**: Data is of type text. There are 1369 unique values. Top 3 frequent values: LH 622, SV 1047, SV 1023. 
**aircraft.model**: Data is of type text. There are 48 unique values. Top 3 frequent values: Airbus A320, Airbus A320 NEO, Airbus A321. 
**aircraft.reg**: Data is of type text. There are 1431 unique values. Top 3 frequent values: HZ-NS34, HZ-NS42, HZ-NS38. 
**aircraft.modeS**: Data is of type text. There are 1612 unique values. Top 3 frequent values: 717CEA, 710DBC, 710DB9. 
**airline.name**: Data is of type text. There are 68 unique values. Top 3 frequent values: Saudi Arabian, flynas, flyadeal. 
**airline.iata**: Data is of type text. There are 62 unique values. Top 3 frequent values: SV, XY, F3. 
**airline.icao**: Data is of type text. There are 63 unique values. Top 3 frequent values: SVA, KNE, FAD. 
**status**: Data is of type text. There are 5 unique values. Top 3 frequent values: Unknown, E

c:\Users\rite\AppData\Local\anaconda3\Lib\site-packages\datamart_profiler\core.py:199: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data = data.astype(object).fillna('').astype(str)
c:\Users\rite\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
c:\Users\rite\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.w